# Shazam (Part 1)

A Running exercise to replicate the Shazam algorithm.

## Goals

- Practicing with pandas `DataFrames`
- Building intuition for distances
- Experimenting with evaluation strategies

## Setup

Run the following 2 cells to import all necessary libraries and helpers for this homework

In [ ]:
!mkdir -p data/audio
!wget -q https://github.com/PSAM-5020-2026F-A/5020-utils/raw/refs/heads/main/datasets/audio/hood-internet.tar.gz
!tar -xzf ./hood-internet.tar.gz -C ./data/audio/ && rm -rf hood-internet.tar.gz*

In [ ]:
import librosa
import matplotlib.pyplot as plt
import pandas as pd

from IPython.display import Audio
from os import listdir

## Shazam !

Yep. Let's replicate the Shazam algorithm. It's the same type of classification task we've been looking at, but instead of classifying a piece of audio into one of $3$ classes, the algorithm has to pick which of the millions of songs the piece of audio most likely came from.

We've created some simple classification models by manually extracting features that we think are important for the differentiation of our classes. This is ok if we're only dealing with a few classes, but it doesn't scale. If we had to classify audio into hundreds, or thousands, of classes, looking at graphs of relative frequencies and loudness for all classes becomes intractable.

We need something a little more systematic. Something that automatically extracts meaningful numeric information from audio, independent of the type of audio.

Let's look at the frequency domain again, and use the librosa library to help us extract consistent frequency information from audio files.

## Librosa Feature Extraction

Librosa is useful because it has many feature extraction functions to actually get meaningful frequency information from our audios files:

[Librosa Reference and Documentation](https://librosa.org/doc/latest/feature.html)

### Some example functions

- `librosa.feature.spectral_centroid` (extracts average frequency)
- `librosa.feature.spectral_bandwidth` (extracts frequency range)
- `librosa.feature.spectral_rolloff(roll_percent=0.01)` (extarcts smallest frequency)
- `librosa.feature.spectral_rolloff(roll_percent=0.99)` (extracts largest frequency)

Let's load a song and see what running these looks like:

### ❗️❗️ WARNING ❗️❗️

The songs we'll be using are from mixtapes by [The Hood Internet](https://www.thehoodinternet.com/). They were part of a wave of _mashup_ producers in the early $2\text{,}000\text{s}$ that mixed _classic_ american indie, pop and rap music from other time periods.

We're using their mixtapes here mostly because: they're available freely on the internet, they encompass a good variety of songs and styles, and there are a good number of songs; not too few nor too many.

The lyrics mostly come from the rap songs being sampled and contain references to all the things that rap artists were rapping about in the $90\text{s}$ and $00\text{s}$ (sex, drugs, guns, money...).

We definitely don't have to listen to the songs to complete the exercise.

We can consider alternative datasets for this exercise. Feel free to adapt the exercise to use files from different albums, just indicate that here in the notebook.

### Back to Features Extraction

In [ ]:
# First, let's grab all audio file names from our album directory

TRACKS_DIR = "./data/audio/hood-internet/mixtape04"
fnames = sorted([f for f in listdir(TRACKS_DIR) if f.endswith("mp3")])

In [ ]:
# opening up a random song (index 4)
y,sr = librosa.load(f"{TRACKS_DIR}/{fnames[4]}")

plt.plot(y)
plt.show()

display(Audio(y, rate=sr))

This is how we extract the center frequency and the lowest/highest frequencies of our signal, throughout the song.

Because these functions were written to work with multi-channel audio, they'll always return a list of lists. In our case, we just have a single channel, so our result will be at index $0$ of the returned list.

That's why we have `sc[0]`, `smin[0]`, `smax[0]` below.

In [ ]:
# average frequency
sc = librosa.feature.spectral_centroid(y=y)

# min and max frequencies
smin = librosa.feature.spectral_rolloff(y=y, roll_percent=0.01)
smax = librosa.feature.spectral_rolloff(y=y, roll_percent=0.99)

plt.plot(sc[0])
plt.plot(smin[0])
plt.plot(smax[0])
plt.show()

This is a start.

The functions give us multiple values because the avg/min/max frequency changes over time, but we can average the `spectral_centroid` and the two `spectral_rolloff` frequencies to get $3$ values that represent our entire audio file.

In [ ]:
sc_val = sum(sc[0]) / len(sc[0])
smin_val = sum(smin[0]) / len(smin[0])
smax_val = sum(smax[0]) / len(smax[0])

plt.plot([0, len(sc[0])], [sc_val, sc_val])
plt.plot([0, len(smin[0])], [smin_val, smin_val])
plt.plot([0, len(smax[0])], [smax_val, smax_val])
plt.show()

Not bad.

### MFCCs

But, we can extend this idea with Mel-Frequency Cepstral Coefficients (MFCC). MFCCs are a compact, numerical representation of an audio signal's frequency characteristics, that divide an audio frequency range into bands based on human hearing sensitivities. So, instead of just $3$ frequencies, like we got above, the total frequency range is divided into $10$ to $20$ values that represent our audio signal.

And of course `librosa` has an `mfcc()` function:

In [ ]:
mfccs = librosa.feature.mfcc(y=y, n_mfcc=10)
print(mfccs.shape)

for vs in mfccs:
  plt.plot(vs)
plt.show()

### Average MFCCs

Like the `spectral_centroid()` function, the `mfcc()` function gives us multiple values for each frequency channel/band, since these change over time as the audio changes. We'll assume that one value per frequency band is enough and average them over time, so we end up with only $10$ numbers to represent our audio.

In [ ]:
mfccs_avg = mfccs.mean(axis=1)
print(mfccs_avg.shape)

for vs in mfccs_avg:
  plt.plot([0,100], [vs, vs])
plt.show()

### Back to Shazam

This is what's behind the [Shazam algorithm](https://www.youtube.com/watch?v=b6xeOLjeKs0): a large database of songs, where each song is described with only a few features so they can be organized, searched, queried and compared very very quickly.

Another [video](https://www.youtube.com/watch?v=kMNSAhsyiDg) reference.

Let's implement something like this.

We'll focus on a single album, with about $30$ songs, but the same strategy (with minor modifications) will work for datasets of many more songs.

Here's the outline of what we have to do:

1. Extract MFCC features from all songs in our database (single album)
2. Create a dataset of song features
3. Given an input sound clip: extract its MFCC features
4. Find closest song in database by feature distance

In [ ]:
# Let's define a helper function that:
#   given a list of samples, 
#   returns a dictionary of extracted mfcc values.
#   to extract the mfcc values, repeat the steps done above:
#     - run mfcc extraction on samples
#     - average each mfcc array
#   the dictionary can have keys in whatever format (mfcc0, mfcc1, ... or feat0, feat1, ...), 
#   we just need named features for the DataFrame
#   check: if n_mfcc is 20, the resulting dict should have 20 keys/values

def get_features(y, n_mfcc=20):
  mfeats = {}
  # TODO: extract/average mfccs, put in dict
  return mfeats

In [ ]:
# TODO: run the get_features() function in one of the loaded files
# TODO: check the output and see if it makes sense

### 1. Extract MFCC features from all songs

It takes about $20$ seconds to extract features for all songs.

In [ ]:
# TODO: Run the function on all tracks and extract their average MFCCs

# TODO: iterate over the list of filenames, open each file,
#       extract mfcc features and append result to a list

track_feats = []

### 2. Create `DataFrame` of song features

In [ ]:
# TODO: put results in pandas DataFrame using the from_records() function

### To Be Continued...

Now we have a dataset of extracted features for all songs in an album.

The idea is that if we repeat the mfcc extraction on a random song clip, we can use distance calculations to find the closest song to the clip.

The next step is to create a function that, given a set of mfcc features, finds the song in our dataset with the most similar features.

This is similar to one of the exercises in Homework02, where we found the closest cities to a given city; and similar to the MoMA exercise, where we found the closest apartment to a location.

The difference is that in those cases our distance was calculated using $2$ features (latitude and longitude), but now our distances will be calculated using $20$ features.

The math is the same. We'll continue next week, but in case you want to write down some notes or test out some stuff....

In [ ]:
# TODO: implement a function that returns distances between 2 mfcc feature lists
def mfcc_dist(mfccA, mfccB):
  sum_of_squares = 0
  # TODO: iterate over pairs of features
  #       calculate the sum of the square of their differences
  #       return the square root of that sum
  return sum_of_squares ** 0.5